# 1. EDA and preprocessing

This notebook examines the exact source frames that cache construction consumes. The raw Liander files remain immutable inputs. A target measurement is available at its measurement timestamp; `available_at` matters only for versioned future-weather forecasts. The standard `vintage` future-weather source is information-safe, whereas `oracle` intentionally supplies future realised weather and is suitable only for a labelled sensitivity analysis.

The final cache builder applies these rules to every forecast origin. This notebook visualises and checks them before the expensive Chronos call.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd() / 'notebooks' if (Path.cwd() / 'notebooks').is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from _helpers import resolve_config
from simcast.cli.build_cache import _load_frames
from simcast.data.grouping import build_entity_group
from simcast.data.availability import select_available_past_targets, select_latest_weather_forecast

CONFIG_FILE = 'configs/liander2024_transformer.yaml'
OVERRIDES: tuple[str, ...] = ()
config = resolve_config(CONFIG_FILE, OVERRIDES)
data_root = Path(config.data.local_dir).expanduser().resolve()
group = build_entity_group(data_root / 'liander2024_targets.yaml', config.data.entity_type)
frames = _load_frames(data_root, group)
print(f'{group.name}: K_g={len(group.entity_ids)}')
display(pd.DataFrame([entity.model_dump() for entity in group.entities]))

## Group integrity and missingness

The ordered IDs below are the group used throughout an experiment. Missingness is reported by entity, but no missing entity is removed. At an invalid `(forecast instance, lead)` the entire vector is invalidated later by `build_group_pit`. This complete-vector rule avoids changing the dimension of the copula problem across cases.

In [ ]:
target_column = config.data.target_column
summary = pd.DataFrame(
    {
        'entity_id': [item.entity.entity_id for item in frames],
        'start': [item.target.index.min() for item in frames],
        'end': [item.target.index.max() for item in frames],
        'target_missing_fraction': [item.target[target_column].isna().mean() for item in frames],
    }
)
display(summary)
assert list(summary['entity_id']) == group.entity_ids
assert len(group.entity_ids) == len(set(group.entity_ids))

## Target and weather availability at one origin

Select a valid timestamp in the first entity frame. Target history is cut at the origin timestamp. Future weather is selected from the latest forecast vintage with `available_at <= origin`; that selection is made independently for each future valid time. The cache builder applies the identical functions to every window.

In [ ]:
entity = frames[0]
origin = entity.target.index[len(entity.target) // 2]
past = entity.target.loc[:origin].tail(config.forecast.lookback_steps)
available_target = select_available_past_targets(past, origin, target_column=target_column)
future_index = pd.date_range(
    origin + pd.Timedelta(minutes=config.forecast.frequency_minutes),
    periods=config.forecast.horizon_steps,
    freq=f"{config.forecast.frequency_minutes}min",
    tz='UTC',
)
weather_vintage = select_latest_weather_forecast(entity.forecast_weather, origin, future_index)
print('origin:', origin)
print('target observations available:', available_target.notna().sum(), '/', len(available_target))
display(weather_vintage[['available_at', *config.covariates.weather]].head())
fig, ax = plt.subplots(figsize=(12, 3))
available_target.plot(ax=ax, label='available target history')
ax.axvline(origin, color='black', linestyle='--', label='forecast origin')
ax.legend(); ax.set_ylabel(target_column); ax.set_title(entity.entity.entity_id)

## Preprocessing contract

`build_cache_from_config` performs the following deterministic sequence: construct aligned origins; prepare each entity's target and covariates; reject an origin if any group member lacks finite required input; obtain frozen Chronos native quantiles and embeddings; apply the configured monotonicity repair; construct finite-cell PIT values; and save an xarray cache with test labels sealed. The next notebook executes that exact function, rather than reproducing this sequence in notebook code.